# DistilBERT Intent Classifier

This notebook trains and evaluates a DistilBERT-based intent classifier
for the ResolveAI customer-support dataset.

The model is trained using the corrected leakage-safe generic-placeholder
training split and evaluated on the corresponding validation split.

The official test split remains untouched during model development.

After model selection, the frozen model will also be evaluated on the
independent OOD challenge benchmark to measure generalization to more
realistic customer language.

In [1]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Using device: cpu


### Model Configuration

In [4]:
MODEL_NAME = "distilbert-base-uncased"

TEXT_COLUMN = "processed_instruction"
TARGET_COLUMN = "intent"

### Load train and validation

In [5]:
train_df = pd.read_csv(
    "../data/processed/generic/train.csv"
)

validation_df = pd.read_csv(
    "../data/processed/generic/validation.csv"
)

print(
    "Train shape:",
    train_df.shape
)

print(
    "Validation shape:",
    validation_df.shape
)

Train shape: (18780, 9)
Validation shape: (4016, 9)


#### Verifying required columns

In [6]:
print(
    train_df.columns.tolist()
)

['flags', 'instruction', 'category', 'intent', 'response', 'normalized_instruction', 'split_group_key', 'processed_instruction', 'split']


In [7]:
required_columns = {
    TEXT_COLUMN,
    TARGET_COLUMN,
}

assert required_columns.issubset(
    train_df.columns
)

assert required_columns.issubset(
    validation_df.columns
)

print(
    "Required columns found."
)

Required columns found.


In [8]:
print(
    train_df[
        [
            TEXT_COLUMN,
            TARGET_COLUMN
        ]
    ]
    .isna()
    .sum()
)

print()

print(
    validation_df[
        [
            TEXT_COLUMN,
            TARGET_COLUMN
        ]
    ]
    .isna()
    .sum()
)

processed_instruction    0
intent                   0
dtype: int64

processed_instruction    0
intent                   0
dtype: int64


#### Check intents

In [9]:
print(
    "Train intents:",
    train_df[TARGET_COLUMN].nunique()
)

print(
    "Validation intents:",
    validation_df[TARGET_COLUMN].nunique()
)

Train intents: 27
Validation intents: 27


In [10]:
sorted(
    train_df[TARGET_COLUMN].unique()
)

['cancel_order',
 'change_order',
 'change_shipping_address',
 'check_cancellation_fee',
 'check_invoice',
 'check_payment_methods',
 'check_refund_policy',
 'complaint',
 'contact_customer_service',
 'contact_human_agent',
 'create_account',
 'delete_account',
 'delivery_options',
 'delivery_period',
 'edit_account',
 'get_invoice',
 'get_refund',
 'newsletter_subscription',
 'payment_issue',
 'place_order',
 'recover_password',
 'registration_problems',
 'review',
 'set_up_shipping_address',
 'switch_account',
 'track_order',
 'track_refund']

### Create ```label2id``` and ```id2label```

In [11]:
# Creating deterministic mapping

intent_labels = sorted(
  train_df[TARGET_COLUMN].unique()
)

label2id = {
  label: idx
  for idx, label in enumerate(
    intent_labels
  )
}

id2label = {
  idx: label
  for label, idx in label2id.items()
}

print(
  "Number of labels: ",
  len(label2id)
)

Number of labels:  27


In [12]:
label2id

{'cancel_order': 0,
 'change_order': 1,
 'change_shipping_address': 2,
 'check_cancellation_fee': 3,
 'check_invoice': 4,
 'check_payment_methods': 5,
 'check_refund_policy': 6,
 'complaint': 7,
 'contact_customer_service': 8,
 'contact_human_agent': 9,
 'create_account': 10,
 'delete_account': 11,
 'delivery_options': 12,
 'delivery_period': 13,
 'edit_account': 14,
 'get_invoice': 15,
 'get_refund': 16,
 'newsletter_subscription': 17,
 'payment_issue': 18,
 'place_order': 19,
 'recover_password': 20,
 'registration_problems': 21,
 'review': 22,
 'set_up_shipping_address': 23,
 'switch_account': 24,
 'track_order': 25,
 'track_refund': 26}

In [13]:
train_df["label"] = (
    train_df[TARGET_COLUMN]
    .map(label2id)
)

validation_df["label"] = (
    validation_df[TARGET_COLUMN]
    .map(label2id)
)

In [14]:
train_df[
    [
        TEXT_COLUMN,
        TARGET_COLUMN,
        "label"
    ]
].head()

,processed_instruction,intent,label
0,question about cancelling order ENTITY_TOKEN,cancel_order,0
1,i need help cancelling puchase ENTITY_TOKEN,cancel_order,0
2,I need to cancel purchase ENTITY_TOKEN,cancel_order,0
3,can you help me cancel order ENTITY_TOKEN?,cancel_order,0
4,"I can no longer afford order ENTITY_TOKEN, can...",cancel_order,0


In [15]:
assert train_df["label"].notna().all()
assert validation_df["label"].notna().all()

assert train_df["label"].nunique() == 27
assert validation_df["label"].nunique() == 27

print(
    "Label encoding validated."
)

Label encoding validated.


## Convert Pandas DataFrames to Hugging Face Datasets

In [16]:
train_dataset = Dataset.from_pandas(
    train_df[
        [
            TEXT_COLUMN,
            "label"
        ]
    ],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df[
        [
            TEXT_COLUMN,
            "label"
        ]
    ],
    preserve_index=False
)

In [17]:
print(train_dataset)
print(validation_dataset)

Dataset({
    features: ['processed_instruction', 'label'],
    num_rows: 18780
})
Dataset({
    features: ['processed_instruction', 'label'],
    num_rows: 4016
})


In [18]:
train_dataset[0]

{'processed_instruction': 'question about cancelling order ENTITY_TOKEN',
 'label': 0}

## Load DistilBERT Tokenizer

In [19]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [20]:
# Inspecting the tokenizer

sample_text = (
    train_df[TEXT_COLUMN]
    .iloc[0]
)

print(sample_text)

question about cancelling order ENTITY_TOKEN


In [21]:
sample_tokens = tokenizer.tokenize(
    sample_text
)

sample_tokens

['question', 'about', 'cancel', '##ling', 'order', 'entity', '_', 'token']

#### Convert tokens to numerical IDs

In [22]:
sample_encoding = tokenizer(
    sample_text
)

sample_encoding

{'input_ids': [101, 3160, 2055, 17542, 2989, 2344, 9178, 1035, 19204, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [23]:
# Decoding the IDs

tokenizer.convert_ids_to_tokens(
    sample_encoding["input_ids"]
)

['[CLS]',
 'question',
 'about',
 'cancel',
 '##ling',
 'order',
 'entity',
 '_',
 'token',
 '[SEP]']

### Check sequence lengths

In [24]:
token_lengths = (
    train_df[TEXT_COLUMN]
    .apply(
        lambda text: len(
            tokenizer.encode(
                text,
                add_special_tokens=True
            )
        )
    )
)

In [25]:
token_lengths.describe(
    percentiles=[
        0.90,
        0.95,
        0.99
    ]
)

count    18780.000000
mean        12.215016
std          3.190224
min          3.000000
90%         16.000000
95%         18.000000
99%         20.000000
max         26.000000
Name: processed_instruction, dtype: float64

In [26]:
print(
    "Maximum token length:",
    token_lengths.max()
)

Maximum token length: 26


### Tokenization function definition

In [27]:
MAX_LENGTH = 64

In [28]:
def tokenize_batch(batch):

    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [29]:
tokenized_train = (
    train_dataset.map(
        tokenize_batch,
        batched=True
    )
)

tokenized_validation = (
    validation_dataset.map(
        tokenize_batch,
        batched=True
    )
)

Map:   0%|          | 0/18780 [00:00<?, ? examples/s]

Map:   0%|          | 0/4016 [00:00<?, ? examples/s]

In [30]:
tokenized_train[0]

{'processed_instruction': 'question about cancelling order ENTITY_TOKEN',
 'label': 0,
 'input_ids': [101, 3160, 2055, 17542, 2989, 2344, 9178, 1035, 19204, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## Dynamic Padding

In [31]:
from transformers import (
    DataCollatorWithPadding,
)

In [32]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

## Load DistilBERT for 27-Class Classification

In [33]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(intent_labels),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
print(
    "Number of labels:",
    model.config.num_labels
)

print(
    "Model labels:",
    model.config.id2label
)

Number of labels: 27
Model labels: {0: 'cancel_order', 1: 'change_order', 2: 'change_shipping_address', 3: 'check_cancellation_fee', 4: 'check_invoice', 5: 'check_payment_methods', 6: 'check_refund_policy', 7: 'complaint', 8: 'contact_customer_service', 9: 'contact_human_agent', 10: 'create_account', 11: 'delete_account', 12: 'delivery_options', 13: 'delivery_period', 14: 'edit_account', 15: 'get_invoice', 16: 'get_refund', 17: 'newsletter_subscription', 18: 'payment_issue', 19: 'place_order', 20: 'recover_password', 21: 'registration_problems', 22: 'review', 23: 'set_up_shipping_address', 24: 'switch_account', 25: 'track_order', 26: 'track_refund'}


## Define Evaluation Metrics

In [35]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

## CPU Training Smoke Test

In [36]:
smoke_train_dataset = (
    tokenized_train
    .shuffle(seed=SEED)
    .select(range(1000))
)

smoke_validation_dataset = (
    tokenized_validation
    .shuffle(seed=SEED)
    .select(range(500))
)

In [37]:
SMOKE_OUTPUT_DIR = (
    "../models/distilbert_smoke_test"
)

In [38]:
smoke_training_args = TrainingArguments(
    output_dir=SMOKE_OUTPUT_DIR,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="no",

    logging_steps=25,

    report_to="none",

    seed=SEED
)

In [39]:
smoke_trainer = Trainer(
    model=model,
    args=smoke_training_args,

    train_dataset=smoke_train_dataset,
    eval_dataset=smoke_validation_dataset,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

### Timed Smoke Test

In [40]:
import time

start_time = time.time()

smoke_train_result = (
    smoke_trainer.train()
)

elapsed_time = (
    time.time()
    - start_time
)

print(
    f"Smoke-test training time: "
    f"{elapsed_time / 60:.2f} minutes"
)

c:\D_Drive\ResolveAI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,3.020629,2.978513,0.254000,0.370269,0.242070,0.196324


Smoke-test training time: 0.87 minutes


In [41]:
smoke_metrics = (
    smoke_trainer.evaluate()
)

smoke_metrics

c:\D_Drive\ResolveAI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
3.020629,2.978513,1,0.254000,0.370269,0.242070,0.196324


{'eval_loss': 2.978513479232788,
 'eval_accuracy': 0.254,
 'eval_macro_precision': 0.3702687884870403,
 'eval_macro_recall': 0.24207013201631392,
 'eval_macro_f1': 0.19632399066981834}

## Full DistilBERT Training

In [42]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(intent_labels),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [43]:
OUTPUT_DIR = "../models/distilbert_intent_classifier"

In [44]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,

    logging_steps=100,

    report_to="none",

    seed=SEED,

    dataloader_pin_memory=False
)

### Full Trainer

In [45]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [46]:
print(
    "Training samples:",
    len(tokenized_train)
)

print(
    "Validation samples:",
    len(tokenized_validation)
)

Training samples: 18780
Validation samples: 4016


In [47]:
start_time = time.time()

train_result = trainer.train()

training_time = (
    time.time()
    - start_time
)

print(
    f"Full training time: "
    f"{training_time / 60:.2f} minutes"
)

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.018309,0.014907,0.995767,0.995803,0.995618,0.995680
2,0.010709,0.014871,0.996763,0.996846,0.996555,0.996667
3,0.000348,0.015231,0.997261,0.997315,0.997119,0.997195


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Full training time: 45.27 minutes


In [48]:
validation_metrics = (
    trainer.evaluate(
        tokenized_validation
    )
)

validation_metrics

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
0.000348,0.015231,3,0.997261,0.997315,0.997119,0.997195


{'eval_loss': 0.015230524353682995,
 'eval_accuracy': 0.9972609561752988,
 'eval_macro_precision': 0.9973151330991766,
 'eval_macro_recall': 0.9971192031052166,
 'eval_macro_f1': 0.9971948449690091}

In [49]:
# Actual Predictions

validation_output = (
    trainer.predict(
        tokenized_validation
    )
)

validation_logits = (
    validation_output.predictions
)

validation_label_ids = (
    validation_output.label_ids
)

distilbert_val_predictions = (
    np.argmax(
        validation_logits,
        axis=-1
    )
)

In [50]:
distilbert_val_intents = [
    id2label[prediction]
    for prediction
    in distilbert_val_predictions
]

true_val_intents = [
    id2label[label]
    for label
    in validation_label_ids
]

In [51]:
print(
    classification_report(
        true_val_intents,
        distilbert_val_intents,
        digits=4
    )
)

                          precision    recall  f1-score   support

            cancel_order     1.0000    0.9688    0.9841       128
            change_order     0.9733    1.0000    0.9865       146
 change_shipping_address     1.0000    1.0000    1.0000       146
  check_cancellation_fee     1.0000    1.0000    1.0000       142
           check_invoice     1.0000    1.0000    1.0000       147
   check_payment_methods     1.0000    1.0000    1.0000       150
     check_refund_policy     1.0000    1.0000    1.0000       149
               complaint     1.0000    1.0000    1.0000       149
contact_customer_service     1.0000    0.9932    0.9966       148
     contact_human_agent     1.0000    1.0000    1.0000       151
          create_account     0.9937    0.9874    0.9905       159
          delete_account     1.0000    0.9926    0.9963       135
        delivery_options     1.0000    1.0000    1.0000       160
         delivery_period     0.9933    1.0000    0.9966       148
         

In [52]:
distilbert_val_errors = (
    distilbert_val_predictions
    != validation_label_ids
).sum()

print(
    "Validation errors:",
    distilbert_val_errors
)

Validation errors: 11


### Evaluate the fromzen Distil BERT on challeng.csv

In [53]:
challenge_df = pd.read_csv(
    "../data/challenge/challenge.csv"
)

In [54]:
challenge_df["label"] = (
    challenge_df["intent"]
    .map(label2id)
)

In [56]:
challenge_dataset = Dataset.from_pandas(
    challenge_df[
        [
            "instruction",
            "label"
        ]
    ],
    preserve_index=False
)

In [57]:
challenge_df["processed_instruction"] = (
    challenge_df["instruction"]
)

In [58]:
challenge_dataset = Dataset.from_pandas(
    challenge_df[
        [
            "processed_instruction",
            "label"
        ]
    ],
    preserve_index=False
)

In [59]:
tokenized_challenge = (
    challenge_dataset.map(
        tokenize_batch,
        batched=True
    )
)

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

In [60]:
challenge_output = (
    trainer.predict(
        tokenized_challenge
    )
)

challenge_logits = (
    challenge_output.predictions
)

challenge_label_ids = (
    challenge_output.label_ids
)

distilbert_challenge_predictions = (
    np.argmax(
        challenge_logits,
        axis=-1
    )
)

In [63]:
from sklearn.metrics import (
    f1_score,
)
challenge_accuracy = accuracy_score(
    challenge_label_ids,
    distilbert_challenge_predictions
)

challenge_macro_f1 = f1_score(
    challenge_label_ids,
    distilbert_challenge_predictions,
    average="macro"
)

challenge_errors = (
    distilbert_challenge_predictions
    != challenge_label_ids
).sum()

print(
    f"Challenge Accuracy: "
    f"{challenge_accuracy:.4f}"
)

print(
    f"Challenge Macro F1: "
    f"{challenge_macro_f1:.4f}"
)

print(
    f"Challenge Errors: "
    f"{challenge_errors}/{len(challenge_df)}"
)

Challenge Accuracy: 0.5333
Challenge Macro F1: 0.5180
Challenge Errors: 126/270


### Performing challenge analysis for DistilBERT

In [64]:
# Building the results data frame

distilbert_challenge_results = (
    challenge_df.copy()
)

distilbert_challenge_results[
    "prediction"
] = [
    id2label[pred]
    for pred
    in distilbert_challenge_predictions
]

distilbert_challenge_results[
    "is_correct"
] = (
    distilbert_challenge_results["intent"]
    ==
    distilbert_challenge_results["prediction"]
)

### Performance by Intent

In [65]:
distilbert_intent_performance = (
    distilbert_challenge_results
    .groupby("intent")
    .agg(
        samples=("is_correct", "size"),
        correct=("is_correct", "sum")
    )
)

distilbert_intent_performance[
    "errors"
] = (
    distilbert_intent_performance["samples"]
    -
    distilbert_intent_performance["correct"]
)

distilbert_intent_performance[
    "accuracy"
] = (
    distilbert_intent_performance["correct"]
    /
    distilbert_intent_performance["samples"]
)

distilbert_intent_performance = (
    distilbert_intent_performance
    .sort_values("accuracy")
)

distilbert_intent_performance

,samples,correct,errors,accuracy
intent,,,,
change_order,10,2,8,0.2
check_refund_policy,10,2,8,0.2
check_payment_methods,10,3,7,0.3
cancel_order,10,3,7,0.3
create_account,10,3,7,0.3
track_order,10,3,7,0.3
payment_issue,10,3,7,0.3
get_refund,10,3,7,0.3
delivery_options,10,4,6,0.4


### Performance by Challenge Type

In [66]:
distilbert_type_performance = (
    distilbert_challenge_results
    .groupby("challenge_type")
    .agg(
        samples=("is_correct", "size"),
        correct=("is_correct", "sum")
    )
)

distilbert_type_performance[
    "errors"
] = (
    distilbert_type_performance["samples"]
    -
    distilbert_type_performance["correct"]
)

distilbert_type_performance[
    "accuracy"
] = (
    distilbert_type_performance["correct"]
    /
    distilbert_type_performance["samples"]
)

distilbert_type_performance = (
    distilbert_type_performance
    .sort_values("accuracy")
)

distilbert_type_performance

,samples,correct,errors,accuracy
challenge_type,,,,
indirect,41,13,28,0.317073
typo,28,13,15,0.464286
distractor,27,13,14,0.481481
colloquial,31,15,16,0.483871
paraphrase,52,29,23,0.557692
long_context,27,16,11,0.592593
real_entity,37,22,15,0.594595
terse,27,23,4,0.851852


### Confusion Pairs

In [67]:
distilbert_errors_df = (
    distilbert_challenge_results[
        ~distilbert_challenge_results[
            "is_correct"
        ]
    ]
)

distilbert_confusion_pairs = (
    distilbert_errors_df
    .groupby(
        [
            "intent",
            "prediction"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)

distilbert_confusion_pairs.head(20)

,intent,prediction,count
67,payment_issue,check_payment_methods,5
32,check_refund_policy,track_refund,4
39,contact_human_agent,contact_customer_service,4
30,check_refund_policy,check_cancellation_fee,3
65,get_refund,track_refund,3
87,track_order,delivery_period,3
50,delivery_options,set_up_shipping_address,3
36,complaint,review,3
6,change_order,change_shipping_address,3
14,change_shipping_address,set_up_shipping_address,3


### DistilBERT Generalization Analysis

DistilBERT achieved slightly lower in-distribution validation accuracy
than Character TF-IDF + Linear SVM (99.73% vs. 99.95%).

However, DistilBERT generalized substantially better to the independent
OOD challenge benchmark, improving accuracy from 44.07% to 53.33% and
Macro-F1 from 41.79% to 51.80%.

The largest gains occurred on paraphrased, long-context, indirect,
colloquial, and terse customer queries. This suggests that pretrained
transformer representations transfer more effectively across changes in
wording than character-level TF-IDF features.

Performance was not uniformly better across every intent. Several
fine-grained semantic boundaries remained difficult, particularly
between related intents such as payment issues vs. payment methods,
refund policy vs. refund tracking, human-agent requests vs. general
customer-service contact, and order tracking vs. delivery-period
questions.

The remaining 46.67% OOD error rate indicates that transformer
architecture alone does not eliminate the distribution-shift problem.
Greater diversity in training data is likely required for robust
production-level intent classification.

## Save Final DistilBERT Model

In [68]:
FINAL_MODEL_DIR = (
    "../models/distilbert_intent_classifier/final"
)

trainer.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

print(
    "Saved DistilBERT model to:",
    FINAL_MODEL_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved DistilBERT model to: ../models/distilbert_intent_classifier/final


In [69]:
import json

label_mapping = {
    "label2id": label2id,
    "id2label": id2label
}

with open(
    "../models/distilbert_intent_classifier/label_mapping.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        label_mapping,
        file,
        indent=4
    )

## Final Findings

DistilBERT achieved approximately 99.73% accuracy and 99.72% Macro-F1
on the leakage-safe Bitext validation split.

Character TF-IDF + Linear SVM achieved slightly higher in-distribution
accuracy at approximately 99.95%.

However, performance differed substantially on the independent
out-of-distribution challenge benchmark:

- Character TF-IDF + Linear SVM: 44.07% accuracy
- DistilBERT: 53.33% accuracy

DistilBERT reduced challenge-set errors from 151 to 126 and produced
especially notable improvements on paraphrased, indirect, long-context,
colloquial, and terse customer queries.

These results suggest that character-level TF-IDF is exceptionally
effective at modelling the lexical structure of the Bitext dataset,
while DistilBERT provides stronger semantic generalization when wording
moves outside the training distribution.

However, the remaining OOD error rate indicates that model architecture
alone does not eliminate distribution shift. Greater diversity in
training data would likely be required for production-level robustness.

No hyperparameters were modified based on challenge-set performance.